# House Price Prediction Model
King County (Seattle area) house sales, 21,613 rows. This notebook runs the whole project in the browser.

In Colab: **Runtime > Run all**. Phase 1 takes under a minute; Phase 2 takes a few minutes on a free Colab CPU.

In [ ]:
!git clone -q https://github.com/aish-19shanbhag/house-price-prediction-model.git
%cd house-price-prediction-model
!pip install -q -r requirements.txt

## 1. The data

In [ ]:
from src.data_loader import load_data
df = load_data()
print(df.shape, df['date'].min().date(), 'to', df['date'].max().date())
df[['price', 'sqft_living', 'bedrooms', 'bathrooms', 'grade', 'zipcode']].describe(include='all').T

## 2. Phase 1: the original 2020 course comparison, reproduced
Simple linear regression, two multiple linear regression feature sets, Lasso (alpha 1/100/1000) and Gradient Boosting on `sqft_living` only, all on the raw price with the notebook's 80/20 split (random_state=3).

In [ ]:
!python -m src.original_comparison

## 3. Phase 2: the 2026 pipeline
Log-price target, engineered features, leakage-free sklearn Pipelines, 5-fold CV plus an untouched 20% hold-out set.

In [ ]:
!python -m src.train

## 4. Results

In [ ]:
import json, pandas as pd
m = json.load(open('reports/metrics/metrics.json'))
rows = {k: {'holdout_rmse': v['holdout']['rmse'], 'holdout_mae': v['holdout']['mae'],
            'holdout_r2': v['holdout']['r2'], 'holdout_adj_r2': v['holdout']['adj_r2'],
            'cv_rmse': f"{v['cv']['rmse_mean']:,.0f} ± {v['cv']['rmse_std']:,.0f}",
            'cv_r2': f"{v['cv']['r2_mean']:.3f} ± {v['cv']['r2_std']:.3f}"} for k, v in m['results'].items()}
print(m['summary'])
pd.DataFrame(rows).T.style.format({'holdout_rmse': '${:,.0f}', 'holdout_mae': '${:,.0f}',
                                   'holdout_r2': '{:.3f}', 'holdout_adj_r2': '{:.3f}'})

In [ ]:
from IPython.display import Image, display
for f in ['phase1_model_comparison', 'phase2_model_comparison', 'predicted_vs_actual', 'residuals',
          'permutation_importance', 'lasso_coefficients', 'correlation_heatmap', 'geo_price_heatmap']:
    display(Image(f'reports/figures/{f}.png'))

### Interactive price map

In [ ]:
from IPython.display import IFrame
IFrame('reports/figures/price_map.html', width='100%', height=520)

## 5. Price a house

In [ ]:
import pandas as pd
from src.predict import predict_frame
sample = pd.read_csv('data/sample_houses.csv', dtype={'zipcode': str})
sample.assign(predicted=predict_frame(sample).round(-3))[
    ['zipcode', 'sqft_living', 'bedrooms', 'grade', 'yr_built', 'price', 'predicted']]